# Librerias

In [ ]:
import pandas as pd            # Manipulación de tablas
import ast                     # Convierte la cadena JSON de 'genres' a lista de dicts
import torch
from langchain.text_splitter import SentenceTransformersTokenTextSplitter
import os

#sklearn
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_validate
from sentence_transformers import SentenceTransformer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.utils.class_weight import compute_class_weight

import numpy as np


### Carga de datos

In [2]:
df_raw = pd.read_csv('data\\movies_metadata.csv', low_memory=False)

print(f"Shape original: {df_raw.shape}")
df_raw.head(2)


Shape original: (45466, 24)


,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


## Paso 1: eliminar sinopsis vacías

In [3]:

df = df_raw.copy()

# Filtrado: overview NOT null y con al menos un caracter alfanumérico
mask_nonempty = df['overview'].notna() & (df['overview'].str.strip() != '')
df = df[mask_nonempty]

print(f"Después del paso 1 (sinopsis no vacías): {df.shape}")


Después del paso 1 (sinopsis no vacías): (44507, 24)


## Paso 2: conservar sólo películas con exactamente un género
Filtramos las filas donde overview está vacío o es NaN, porque no aportan texto para los embeddings.

In [4]:
def parse_genres(genre_str):
    """Convierte la cadena JSON de 'genres' en lista de nombres de géneros."""
    try:
        items = ast.literal_eval(genre_str) if pd.notna(genre_str) else []
        return [d['name'] for d in items]
    except (ValueError, SyntaxError):
        return []

# Convertir y almacenar la lista en una columna auxiliar
df['genres_list'] = df['genres'].apply(parse_genres)

# Quedarnos con películas que tengan exactamente 1 género
df_single = df[df['genres_list'].str.len() == 1].copy()

# Crear columna 'genre' con ese único valor
df_single['genre'] = df_single['genres_list'].str[0]

print(f"Después del paso 2 (un solo género): {df_single.shape}")
df_single['genre'].value_counts().head()


Después del paso 2 (un solo género): (14136, 26)


genre
Drama          4909
Comedy         3371
Documentary    2685
Horror          971
Thriller        455
Name: count, dtype: int64

## Paso 3: filtrar por la lista de géneros válidos

In [5]:
VALID_GENRES = [
    'Drama', 'Comedy', 'Documentary', 'Horror', 'Thriller', 'Western',
    'Action', 'Animation', 'Science Fiction', 'Crime', 'Music', 'Adventure'
]

df_valid = df_single[df_single['genre'].isin(VALID_GENRES)].copy()

print(f"Después del paso 3 (géneros válidos): {df_valid.shape}")
df_valid['genre'].value_counts()


Después del paso 3 (géneros válidos): (13759, 26)


genre
Drama              4909
Comedy             3371
Documentary        2685
Horror              971
Thriller            455
Western             317
Action              276
Animation           237
Science Fiction     195
Crime               127
Music               109
Adventure           107
Name: count, dtype: int64

## Paso 4: codificar el género en la columna

In [6]:
# Diccionario género: código numérico (orden alfabético para reproducibilidad)
genre2code = {g: idx for idx, g in enumerate(sorted(VALID_GENRES))}
df_valid["genre_code"] = df_valid["genre"].map(genre2code)

# Inspección rápida
print("Diccionario género → código:", genre2code)
df_valid[["genre", "genre_code"]].head()


Diccionario género → código: {'Action': 0, 'Adventure': 1, 'Animation': 2, 'Comedy': 3, 'Crime': 4, 'Documentary': 5, 'Drama': 6, 'Horror': 7, 'Music': 8, 'Science Fiction': 9, 'Thriller': 10, 'Western': 11}


,genre,genre_code
4,Comedy,3
25,Drama,6
35,Drama,6
39,Drama,6
54,Drama,6


## Paso 5: filtrar sinopsis que se dividen en más de un chunk

In [9]:

splitter = SentenceTransformersTokenTextSplitter(
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        tokens_per_chunk=256,
        chunk_overlap=0
)

mask_one_chunk = df_valid["overview"].apply(
        lambda txt: len(splitter.split_text(str(txt))) == 1
)
df_chunk = df_valid[mask_one_chunk].copy()

print(f"Después del paso 5 (sinopsis de un solo bloque): {df_chunk.shape}")


Después del paso 5 (sinopsis de un solo bloque): (13745, 27)


## Paso 6: codificar el género y reducir a 4 columnas

In [11]:

genre2code = {g: idx for idx, g in enumerate(sorted(VALID_GENRES))}
df_chunk["genre_code"] = df_chunk["genre"].map(genre2code)

# Nos quedamos solamente con las columnas requeridas
df_final = df_chunk[["title", "overview", "genre", "genre_code"]].copy()

print(f"Después del paso 6 (columnas reducidas): {df_final.shape}")
df_final.head(2)


Después del paso 6 (columnas reducidas): (13745, 4)


,title,overview,genre,genre_code
4,Father of the Bride Part II,Just when George Banks has recovered from his ...,Comedy,3
25,Othello,The evil Iago pretends to be friend of Othello...,Drama,6


## Paso 7: verificar tamaño final (13 744 × 4)

In [15]:

expected_shape = (13745, 4)
assert df_final.shape == expected_shape, \
       f"Tamaño inesperado: {df_final.shape}, se esperaban {expected_shape}"

print("  Paso 7 superado — dataset final con 13 744 observaciones y 4 variables.")


  Paso 7 superado — dataset final con 13 744 observaciones y 4 variables.


## Paso 8: división 80 / 20 en entrenamiento y prueba

In [16]:

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        df_final["overview"],
        df_final["genre_code"],
        test_size=0.20,
        stratify=df_final["genre_code"],
        random_state=42
)

print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")


Train: 10996  |  Test: 2749


## Crear el embedder y los pipelines

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embedder = FunctionTransformer(
    lambda texts: model.encode(list(texts), batch_size=64,
                               show_progress_bar=False,
                               normalize_embeddings=True),
    validate=False
)

# Pipelines base ------------------------------------------------------
pipelines = {
    "LogReg": Pipeline([("emb", embedder),
                        ("clf", LogisticRegression(max_iter=2000,
                                                   class_weight="balanced",
                                                   n_jobs=-1,
                                                   multi_class="ovr"))]),
    "LinearSVC": Pipeline([("emb", embedder),
                           ("clf", LinearSVC(class_weight="balanced"))]),
    "RandomForest": Pipeline([("emb", embedder),
                              ("clf", RandomForestClassifier(
                                  n_jobs=-1,
                                  class_weight="balanced_subsample",
                                  random_state=42))]),
    "MLP": Pipeline([("emb", embedder),
                     ("clf", MLPClassifier(max_iter=300,
                                           early_stopping=True,
                                           random_state=42))])
}

# Espacios de hiperparámetros (RandomizedSearch) ----------------------
param_dists = {
    "LogReg": {
        "clf__C": [0.1, 1, 10, 50],
        "clf__solver": ["lbfgs", "sag", "newton-cg"]
    },
    "LinearSVC": {
        "clf__C": [0.01, 0.1, 1, 10, 50]
    },
    "RandomForest": {
        "clf__n_estimators": [200, 400, 600],
        "clf__max_depth": [None, 30, 60],
        "clf__min_samples_leaf": [1, 2, 4]
    },
    "MLP": {
        "clf__hidden_layer_sizes": [(256,), (512,), (512, 256)],
        "clf__alpha": [1e-4, 1e-3, 1e-2],
        "clf__learning_rate_init": [1e-3, 5e-4]
    }
}

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"accuracy": make_scorer(accuracy_score),
           "f1_macro": make_scorer(f1_score, average="macro")}

results = []
for name in tqdm(pipelines, desc="Modelos"):           # barra de progreso exterior
    pipe = pipelines[name]
    search = RandomizedSearchCV(
        pipe,
        param_distributions=param_dists[name],
        n_iter=6,                # número de combinaciones a probar
        scoring=scoring,
        cv=cv,
        refit="f1_macro",        # refit con la métrica objetivo
        n_jobs=-1,
        verbose=0,
        random_state=42,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    best_scores = search.cv_results_
    results.append({
        "model": name,
        "best_params": search.best_params_,
        "acc_mean": search.best_score_ if search.scorer_ == scoring["accuracy"] else
                    best_scores["mean_test_accuracy"][search.best_index_],
        "f1_mean": search.best_score_,   # porque refit=f1_macro
    })
    print(f"{name:10s}  F1={search.best_score_:.3f}  Acc={best_scores['mean_test_accuracy'][search.best_index_]:.3f}")

cv_df = pd.DataFrame(results).sort_values("f1_mean", ascending=False)
display(cv_df[["model", "f1_mean", "acc_mean"]])

# 3️  Entrenar pipeline óptimo en todos los datos de entrenamiento ----
best_name = cv_df.iloc[0]["model"]
best_pipe = pipelines[best_name].set_params(**cv_df.iloc[0]["best_params"])
best_pipe.fit(X_train, y_train)
print(f"\n● Mejor modelo final: {best_name}")

# 4️⃣  Evaluación en el set de prueba ---------------------------------
y_pred = best_pipe.predict(X_test)
print("\n=== Reporte en test ===")
print(classification_report(y_test, y_pred, target_names=sorted(genre2code.keys())))
print(f"Accuracy (test): {accuracy_score(y_test, y_pred):.3f}")
print(f"F1‑macro (test): {f1_score(y_test, y_pred, average='macro'):.3f}")

LogReg      Acc=0.610±0.006  F1=0.457±0.011
